# Umba Fraud Detection — Data Exploration
Exploratory analysis of `train.csv`, `test.csv`, and `identity.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA = 'data/raw/'
train    = pd.read_csv(DATA + 'train.csv')
test     = pd.read_csv(DATA + 'test.csv')
identity = pd.read_csv(DATA + 'identity.csv')

print(f'Train:    {train.shape}')
print(f'Test:     {test.shape}')
print(f'Identity: {identity.shape}')

## 1. Basic shapes & dtypes

In [ ]:
print('=== TRAIN ===')
print(train.dtypes.value_counts())
print('\n=== TEST ===')
print(test.dtypes.value_counts())
train.head(3)

## 2. Missing values

In [ ]:
def missing_report(df, name):
    miss = df.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    pct  = (miss / len(df) * 100).round(1)
    return pd.DataFrame({'missing': miss, 'pct': pct}).rename_axis(name)

print('=== TRAIN ===')
display(missing_report(train, 'column').head(20))
print('\n=== TEST ===')
display(missing_report(test, 'column').head(20))

## 3. Target distribution (train only)

In [ ]:
vc = train['isFraud'].value_counts()
print(vc)
print(f'\nFraud rate: {vc[1]/len(train)*100:.2f}%')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Legit (0)', 'Fraud (1)'], vc.values, color=['#3b82f6', '#f87171'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 4. Temporal split verification

In [ ]:
print(f'Train TransactionDT:  min={train.TransactionDT.min():,}  max={train.TransactionDT.max():,}')
print(f'Test  TransactionDT:  min={test.TransactionDT.min():,}  max={test.TransactionDT.max():,}')
overlap = test.TransactionDT.min() <= train.TransactionDT.max()
print(f'\nTemporal overlap: {overlap}  (should be False — test must come AFTER train)')

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(train.TransactionDT, bins=100, alpha=0.6, label='Train', color='#3b82f6')
ax.hist(test.TransactionDT,  bins=100, alpha=0.6, label='Test',  color='#f87171')
ax.set_title('TransactionDT Distribution')
ax.set_xlabel('TransactionDT (seconds offset)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Transaction amount analysis

In [ ]:
print('=== TransactionAmt stats ===')
print(train.groupby('currency')['TransactionAmt'].describe().round(2))

# KES vs NGN scale
KES_TO_USD = 1/128
NGN_TO_USD = 1/1570
train['amt_usd'] = train.apply(
    lambda r: r.TransactionAmt * (KES_TO_USD if r.currency == 'KES' else NGN_TO_USD), axis=1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, currency, color in zip(axes, ['KES', 'NGN'], ['#3b82f6', '#f59e0b']):
    d = train[train.currency == currency]['TransactionAmt']
    ax.hist(np.log1p(d), bins=50, color=color, alpha=0.8)
    ax.set_title(f'log(1+Amount) — {currency}')
    ax.set_xlabel('log(1 + TransactionAmt)')
plt.tight_layout()
plt.show()

## 6. Fraud rate by categorical features

In [ ]:
cat_cols = ['country', 'currency', 'channel', 'card_type', 'card_bank']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 4))
for ax, col in zip(axes, cat_cols):
    fraud_rate = train.groupby(col)['isFraud'].mean().sort_values(ascending=False)
    fraud_rate.plot(kind='bar', ax=ax, color='#f87171', alpha=0.8)
    ax.set_title(f'Fraud rate by {col}')
    ax.set_ylabel('Fraud rate')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 7. Time-of-day & day-of-week patterns

In [ ]:
train['hour'] = (train.TransactionDT // 3600) % 24
train['dow']  = (train.TransactionDT // 86400) % 7

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Hour of day
h = train.groupby('hour')['isFraud'].agg(['mean', 'count'])
axes[0].bar(h.index, h['mean'], color='#3b82f6', alpha=0.8)
axes[0].set_title('Fraud rate by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Fraud rate')

# Day of week
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
d = train.groupby('dow')['isFraud'].mean()
axes[1].bar([days[i] for i in d.index], d.values, color='#a78bfa', alpha=0.8)
axes[1].set_title('Fraud rate by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Fraud rate')

plt.tight_layout()
plt.show()

## 8. Amount distribution: fraud vs legit

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [(0, '#3b82f6'), (1, '#f87171')]:
    d = train[train.isFraud == label]['TransactionAmt']
    ax.hist(np.log1p(d), bins=80, alpha=0.5,
            label=f'{"Legit" if label==0 else "Fraud"} (n={len(d):,})', color=color)
ax.set_title('log(1+Amount): Fraud vs Legit')
ax.set_xlabel('log(1 + TransactionAmt)')
ax.legend()
plt.tight_layout()
plt.show()

print('=== Amount stats by class ===')
print(train.groupby('isFraud')['TransactionAmt'].describe().round(2))

## 9. Identity join analysis

In [ ]:
print(f'Identity rows: {len(identity):,}')
print(f'Train rows:    {len(train):,}')
print(f'Identity match rate: {identity.TransactionID.isin(train.TransactionID).mean()*100:.1f}%')
print(f'Train with identity: {train.TransactionID.isin(identity.TransactionID).mean()*100:.1f}%')

# Multi-row check
multi = identity.groupby('TransactionID').size()
print(f'\nIdentity multi-row transactions: {(multi>1).sum():,} ({(multi>1).mean()*100:.1f}%)')

identity.head(3)

## 10. Leakage check — flagged_for_review

In [ ]:
if 'flagged_for_review' in train.columns:
    print('WARNING: flagged_for_review is in train — check for leakage!')
    print(train['flagged_for_review'].value_counts(dropna=False))
    corr = train['flagged_for_review'].fillna(0).corr(train['isFraud'])
    print(f'\nCorrelation with isFraud: {corr:.4f}')
    print('\nThis feature MUST be excluded from model training.')
else:
    print('flagged_for_review not found in train columns.')

## 11. Correlation heatmap (numeric features)

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
# Keep only cols with low missingness
num_cols = [c for c in num_cols if train[c].isnull().mean() < 0.5 and c != 'flagged_for_review']

corr = train[num_cols].corr()['isFraud'].drop('isFraud').sort_values(key=abs, ascending=False)
top = corr.head(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#f87171' if v > 0 else '#3b82f6' for v in top.values]
top.plot(kind='barh', ax=ax, color=colors, alpha=0.8)
ax.set_title('Top 20 features correlated with isFraud')
ax.set_xlabel('Pearson correlation')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 12. Train vs Test feature distribution check

In [ ]:
# Compare key numeric features between train and test
check_cols = ['TransactionAmt', 'recipient_account_age_days', 'sender_prev_txn_count']
check_cols = [c for c in check_cols if c in train.columns and c in test.columns]

fig, axes = plt.subplots(1, len(check_cols), figsize=(14, 4))
if len(check_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, check_cols):
    ax.hist(np.log1p(train[col].dropna()), bins=50, alpha=0.5, label='Train', color='#3b82f6')
    ax.hist(np.log1p(test[col].dropna()),  bins=50, alpha=0.5, label='Test',  color='#f87171')
    ax.set_title(f'log(1+{col})')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 13. Summary statistics

In [ ]:
print('=== TRAIN SUMMARY ===')
display(train.describe(include='all').T[['count','mean','std','min','max']].head(30))
print('\n=== TEST SUMMARY ===')
display(test.describe(include='all').T[['count','mean','std','min','max']].head(30))